### Этап 1: Инициализация и очистка

In [1]:
# Импорт необходимых модулей и подключение к базе данных, чтение очищенного представления(view)

import pandas as pd
import sqlite3
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
import re
from collections import defaultdict
conn = sqlite3.connect('../SQL/Kickstarter_db.db')
df = pd.read_sql_query("SELECT * FROM KS_Projects_Modified", conn)
conn.close()

In [2]:
# Преобразование дат в корректный формат

df['launched'] = pd.to_datetime(df['launched'])
df['deadline'] = pd.to_datetime(df['deadline'])

In [3]:
# Проверка ID на дубликаты

print("ID duplicates: ",df['ID'].duplicated().sum())

ID duplicates:  0


In [4]:
# Введение вспомогательных столбцов

df['is_successful'] = df['state'] == 'successful'
df['duration'] = (df['deadline'] - df['launched']).dt.days

In [5]:
# Получаем общую информацию о датафрейме

df.info()
df.sample(5)

<class 'pandas.DataFrame'>
RangeIndex: 319321 entries, 0 to 319320
Data columns (total 15 columns):
 #   Column         Non-Null Count   Dtype         
---  ------         --------------   -----         
 0   ID             319321 non-null  int64         
 1   name           319321 non-null  str           
 2   category       319321 non-null  str           
 3   main_category  319321 non-null  str           
 4   currency       319321 non-null  str           
 5   deadline       319321 non-null  datetime64[us]
 6   goal           319321 non-null  float64       
 7   launched       319321 non-null  datetime64[us]
 8   pledged        319321 non-null  float64       
 9   state          319321 non-null  str           
 10  backers        319321 non-null  int64         
 11  country        319321 non-null  str           
 12  usd pledged    319321 non-null  float64       
 13  is_successful  319321 non-null  bool          
 14  duration       319321 non-null  int64         
dtypes: bool(1),

,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged,is_successful,duration
3596,1021779971,Mississippi River Expedition: G'wine Back!,Music Videos,Film & Video,USD,2015-04-01,5000.0,2015-03-09,6572.0,successful,114,US,6572.0,True,23
132530,1798346162,The Exile Nation Project: a film by Charles Shaw,Film & Video,Film & Video,USD,2011-06-01,7000.0,2011-03-02,7771.0,successful,91,US,7771.0,True,91
309788,941887632,The Shelter (short film),Shorts,Film & Video,USD,2012-01-08,20000.0,2011-11-09,780.0,failed,5,US,780.0,False,60
34024,1204982648,LaserWand Magic Game,Gaming Hardware,Games,EUR,2016-12-11,50.0,2016-11-22,0.0,live,0,DE,0.0,False,19
127258,1765739781,"Between Here and Now, a new dance work by Ande...",Performances,Dance,USD,2016-04-28,3500.0,2016-04-15,3783.0,successful,57,US,3783.0,True,13


In [6]:
# Получаем статистическую информацию о числовых столбцах

pd.set_option('display.float_format', '{:.2f}'.format)
df[['goal','usd pledged','backers']].describe()

,goal,usd pledged,backers
count,319321.00,319321.00,319321.00
mean,47648.81,7847.85,102.84
std,1146316.33,84684.97,940.39
min,0.01,0.00,0.00
25%,2000.00,25.00,2.00
50%,5000.00,535.00,12.00
75%,15000.00,3575.00,56.00
max,100000000.00,20338986.27,219382.00


In [7]:
# Обрабатываем выбросы, оставляя 99 перцентиль.

n1 = df['goal'].count()

df = df[
    (df['usd pledged'] <= df['usd pledged'].quantile(0.99)) &
    (df['goal'] <= df['goal'].quantile(0.99)) &
    (df['backers'] <= df['backers'].quantile(0.99))
]

n2 = df['goal'].count()

print("Убрано ", n1-n2, " выбросов")
df[['goal','usd pledged','backers']].describe()

Убрано  7463  выбросов


,goal,usd pledged,backers
count,311858.00,311858.00,311858.00
mean,16452.74,4044.36,57.11
std,34158.11,9739.83,131.17
min,0.01,0.00,0.00
25%,2000.00,25.00,2.00
50%,5000.00,517.00,12.00
75%,15000.00,3360.00,53.00
max,374484.00,104884.00,1420.00


### Этап 2: Исследовательский анализ

#### Анализ успешности

In [8]:
# 1. Общая доля успешных проектов

print(f"Общая доля успешных проектов: {round(df["ID"][df["state"] == 'successful'].count() / df["ID"].count() * 100,2)}%")

# Вывод: 
# Общая доля успешных проектов равна 34.88%

Общая доля успешных проектов: 34.88%


In [9]:
# 2. Доля успешных проектов по категориям

category_success = df.groupby('main_category', as_index=False).agg(
    total_projects = ('state','count'),
    successful_projects = ('is_successful','sum'),
    goal_median = ('goal','median'),
    pledged_median = ('usd pledged','median')
)

category_success['success_rate_%'] = (category_success['successful_projects'] / category_success['total_projects'])*100

category_success.sort_values('success_rate_%',ascending=False)

# Вывод: 
# Топ-3 наиболее успешных категории: 
# 1."Танцы" 
# 2."Театр" 
# 3."Комиксы"


,main_category,total_projects,successful_projects,goal_median,pledged_median,success_rate_%
3,Dance,3357,2100,3000.00,1710.00,62.56
14,Theater,9884,5975,3000.00,1500.00,60.45
1,Comics,8603,4374,3600.00,1192.00,50.84
10,Music,44126,21585,4000.00,906.00,48.92
0,Art,23772,9608,3000.00,391.00,40.42
6,Film & Video,55552,21048,6000.00,709.17,37.89
8,Games,26357,8201,8000.00,815.00,31.12
4,Design,22607,6952,10000.00,1310.00,30.75
12,Publishing,33539,10153,5000.00,232.00,30.27
11,Photography,9629,2888,3900.00,201.00,29.99


In [10]:
# 3. Доля успешных проектов по странам

country_success = df.groupby('country', as_index=False).agg(
    total_projects = ('state','count'),
    successful_projects = ('is_successful','sum')
)

country_success['success_rate_%'] = (country_success['successful_projects'] / country_success['total_projects'])*100

country_success.sort_values('success_rate_%',ascending=False)

# Вывод: 
# Топ-3 наиболее успешных стран по сбору средств: 
# 1. США 
# 2. Великобритания 
# 3. Люксембург 

,country,total_projects,successful_projects,success_rate_%
20,US,251721,92216,36.63
9,GB,27018,9045,33.48
13,LU,40,13,32.50
6,DK,751,223,29.69
17,NZ,1119,326,29.13
18,SE,1118,322,28.80
19,SG,117,31,26.50
8,FR,1835,474,25.83
3,CA,11708,2941,25.12
11,IE,560,130,23.21


In [11]:
# Анализ "качества" спонсоров по категориям

df_backer_q = df[['main_category','usd pledged', 'backers']]
df_backer_q = df_backer_q.groupby('main_category', as_index=False).agg(
    avg_backers = ('backers', 'mean'),
    avg_pledged = ('usd pledged', 'mean'),
)
df_backer_q['pledged_per_backer'] = (df_backer_q['avg_pledged'] / df_backer_q['avg_backers']).fillna(0)

df_backer_q = df_backer_q.sort_values('pledged_per_backer', ascending=False)
df_backer_q

# Вывод:
# В категориях "Фильмы и видео" и "Технологии" спонсоры готовы вкладывать наибольшее кол-во средств, в данных категориях приходится более $90 на одного спонсора.
# В категориях "Игры" и "Комиксы" спонсор, в среднем, готов вложить не более $50, что является наименьшим показателем из всех категорий.
# Стоит также отметить пропорциональное изменение среднего кол-ва спонсоров между высшей и низшей категорией данного рейтинга, 
# этот факт приблизительно уравновешивает итоговое среднее кол-во привлеченных средств. 

,main_category,avg_backers,avg_pledged,pledged_per_backer
6,Film & Video,49.07,4607.33,93.90
13,Technology,59.30,5466.61,92.19
7,Food,44.71,3804.70,85.10
5,Fashion,40.32,3405.76,84.48
14,Theater,45.00,3587.06,79.71
3,Dance,42.18,3234.23,76.67
11,Photography,34.38,2573.13,74.85
4,Design,100.83,7203.94,71.45
10,Music,48.68,3376.15,69.35
0,Art,35.64,2439.87,68.46


#### Анализ длительности кампании

In [12]:
# 1. Наиболее часто встречающиеся длительности кампаний

duration_count = df.groupby('duration').agg(
    Count = ('ID','count')
)
duration_count["Ratio_%"] = (duration_count["Count"] / df["ID"].count()) * 100
duration_count.sort_values('Count',ascending=False).head(10)

# Вывод:
# Наиболее популярная длительность кампании - 30 дней

,Count,Ratio_%
duration,,
30,139528,44.74
60,27360,8.77
45,14556,4.67
31,10893,3.49
40,8200,2.63
35,8099,2.60
32,6019,1.93
20,5511,1.77
21,5487,1.76


In [13]:
# 2. Связь длительности кампании и её успеха

duration_df = df[df["state"].isin(['successful','failed','canceled'])]

duration_df['duration_range'] = pd.cut(
    duration_df['duration'],
    bins=[0,30,60,90,120],
    labels=['1-30 days','31-60 days','61-90 days', '91+ days']
    )

duration_df_grouped = duration_df.groupby('duration_range').agg(
    Count = ('ID','count'),
    Success_Count = ('is_successful','sum')
    )
duration_df_grouped['Duration_Ratio_%'] = (duration_df_grouped['Count'] / duration_df["ID"].count())*100
duration_df_grouped['Success_Ratio_In_Range_%'] = (duration_df_grouped['Success_Count'] / duration_df_grouped["Count"])*100


duration_df_grouped

# Вывод:
# Наиболее успешный диапазоны длительности кампании - 1-30 дней

,Count,Success_Count,Duration_Ratio_%,Success_Ratio_In_Range_%
duration_range,,,,
1-30 days,192816,70167,62.99,36.39
31-60 days,107917,36772,35.25,34.07
61-90 days,4907,1706,1.60,34.77
91+ days,483,141,0.16,29.19


#### Корреляционный анализ

In [14]:
# Кореляционный анализ успешности проекта методом Пирсона функцией .corr

df_corr = df[['is_successful', 'goal', 'usd pledged', 'duration','backers']]
df_corr['pledged_per_backer'] = (df['usd pledged'] / df['backers']).fillna(0)
df_corr_matrix = df_corr.corr()
df_corr_matrix = df_corr_matrix['is_successful'].sort_values(ascending=False)
df_corr_matrix

# Вывод: 
# Наиболее сильные предикторы успеха: 
# 1.Кол-во спонсоров 
# 2.Привлеченные средства 
# Также наблюдается умеренная отрицательная связь с величиной цели сбора, чем выше цель, тем ниже вероятность успеха. 

is_successful         1.00
backers               0.41
usd pledged           0.39
pledged_per_backer    0.14
duration             -0.11
goal                 -0.20
Name: is_successful, dtype: float64

#### Текстовый анализ

In [15]:
# Рейтинг слов по величине сбора средств и успешности проекта в категории "Технологии"

# Инициализация, определение "мусорных" слов.

df_text = df[df['state'].isin(['successful', 'failed', 'canceled']) & (df['main_category'] == 'Technology')].copy()

stop_words = {
    
    'the', 'a', 'an', 'and', 'of', 'to', 'for', 'in', 'on', 'at', 
    'with', 'by', 'is', 'are', 'was', 'were', 'be', 'been', 'being',
    'this', 'that', 'these', 'those', 'it', 'they', 'we', 'you','your', 'he', 'she',
    'from', 'or', 'as', 'but', 'not', 'can', 'will', 'just', 'so',
    'out', 'up', 'down', 'into', 'than', 'then', 'now', 'has', 'have',
    'had', 'do', 'does', 'did', 'would', 'could', 'should', 'may', 'might',
    
    
    '-', '—', '–', '+', '=', '&', '|', '#', '@', '$', '%', '^', '*', '(', ')',
    '[', ']', '{', '}', '<', '>', '/', '\\', '|', '~', '`', 
    '0', '1', '2', '3', '4', '5', '6', '7', '8', '9', '10', '11', '12',
 
    '(canceled)', 'canceled'
}







In [16]:
# Вычислительная часть

word_stats = defaultdict(lambda: {'count': 0, 'success_sum': 0})

for _, row in df_text.iterrows():
    if pd.isna(row['name']):
        continue

    words = set(str(row['name']).lower().split())
    

    for w in words:
        if w not in stop_words:
            word_stats[w]['count'] += 1
            word_stats[w]['success_sum'] += row['is_successful']


df_word_stats = pd.DataFrame.from_dict(word_stats, orient='index').reset_index()
df_word_stats['success_rate_%'] = 100 * df_word_stats['success_sum'] / df_word_stats['count']



In [17]:
# Вывод результата (берём слова, которые встречаются не менее 100 раз)

df_word_stats = df_word_stats[df_word_stats['count'] > 100]
df_word_stats = df_word_stats.sort_values('success_rate_%',ascending=False)
df_word_stats.head(20)

,index,count,success_sum,success_rate_%
980,raspberry,196,126,64.29
977,pi,196,121,61.73
136,arduino,211,117,55.45
399,board,121,63,52.07
534,robotics,115,58,50.43
104,camera,124,54,43.55
507,kit,156,66,42.31
212,learn,151,61,40.40
871,robot,189,72,38.10
1597,most,113,42,37.17


In [18]:
# Образец случайных проектов включяющих одно или несколько слов из топ-3

word_top = df_word_stats['index'].head(3).tolist()
df_word_stats_sample = df[
    (df['main_category'] == 'Technology') & (
    (df['name'].str.contains(r'\b' + word_top[0] + r'\b', case=False)) |
    (df['name'].str.contains(r'\b' + word_top[1] + r'\b', case=False)) |
    (df['name'].str.contains(r'\b' + word_top[2] + r'\b', case=False))
    ) 
    ]

df_word_stats_sample.sample(5)



,ID,name,category,main_category,currency,deadline,goal,launched,pledged,state,backers,country,usd pledged,is_successful,duration
223581,417164296,The AWESOME Arduino Prototyping Set & Bootload...,DIY Electronics,Technology,CAD,2015-05-16,800.00,2015-04-16,2905.00,successful,56,CA,2324.29,True,30
136498,1821928819,URUK WiFi Router Module (Arduino Compatible),Hardware,Technology,USD,2013-07-10,10000.00,2013-06-10,6807.00,failed,158,US,6807.00,False,30
87320,1526192434,SparqEE CELLv1.0: Cellular made easy (Arduino/...,DIY Electronics,Technology,USD,2013-09-19,70000.00,2013-08-20,72155.00,successful,446,US,72155.00,True,30
248446,56800541,Pi-powered Experimenter Bench --PEB (Canceled),Technology,Technology,USD,2013-03-31,50000.00,2013-03-01,3703.00,canceled,45,US,3703.00,False,30
263503,66038634,Raspberry Pi Network Configurator � PiConfig,Software,Technology,GBP,2015-04-20,650.00,2015-03-21,2191.00,successful,195,GB,3269.91,True,30


In [19]:
df_word_stats_sample[['goal','usd pledged', 'backers']].describe()

,goal,usd pledged,backers
count,480.00,480.00,480.00
mean,13133.78,9311.82,145.26
std,25252.94,15492.78,216.63
min,1.00,0.00,0.00
25%,1500.00,1049.75,20.75
50%,4500.00,3276.12,70.50
75%,12125.00,9316.25,162.25
max,200000.00,103748.00,1320.00


In [20]:
# Вывод:
# Наиболее успешны в категории "Технологии" проекты, которые связаны с Raspberry pi и Arduino, более чем в 50% случаях им удается привлечь целевое кол-во средств.

### Этап 3: Выгрузка результирующих CSV

In [21]:
df.to_csv('CSV_Output/kickstarter_bi_clean.csv', index=False, encoding='utf-8')
df_corr_matrix.to_csv('CSV_Output/kickstarter_corr_matrix.csv', index=True, encoding='utf-8')